# 06o — Generalization: BPI-Challenge-2012 (process-mining domain, Phase 3) — PRIMARY

**PAPER_TODO §2.2.** ~13k loan-application traces over **one shared activity vocabulary** — a
cross-domain, at-scale test of whether symbol *order* discriminates the outcome beyond *frequency*.
The SDS2 cohort's order-null is 0/15 (RH §12); the action-segmentation family is
frequency-*saturated* (06m/06m2/06m3, `headroom.py`). BPI-2012 was the prime candidate for a
genuinely frequency-controlled real order test.

**Task frame — minimal leakage guard.** Symbol = activity `concept:name` (COMPLETE-lifecycle
events only); label = terminal outcome (`A_APPROVED`/`A_DECLINED`/`A_CANCELLED`). The three
outcome-defining activities are **removed from the input** — they are activities *inside* each
trace, so keeping them makes both the headroom screen and the order-null trivially `AUC=1.0` by
label definition (leakage, distinct from frequency saturation).

**Result (this notebook).**
1. **Frequency saturates** the outcome at every face-value level: full-vocab `hist_auc` ≥ 0.986,
   and even the shared-vocabulary sub-tasks ≥ 0.971. BPI-2012 **joins Breakfast** — frequency
   dominates, now in a new domain (process mining) and at scale (12.7k traces).
2. **Depth:** the saturation is driven by trace *length* (declined traces are short). Controlling
   length opens headroom on the two "declined" contrasts (`hist_auc` 0.74 / 0.90) — a **constructed
   frequency-AND-length-controlled order test**. §4 runs a bounded order-null there with the
   mandatory triple gate (calibration decoy, negative control, mechanism); the §15 shuffle-null is
   anti-conservative when frequency is present, so a positive is claimed only if all three pass.

Bounds: order-null on the length-matched sweet pairs at ≤200/class, `n_shuffles`=100 (4 feature×shuffle
combos); quality half on all 3 outcomes at ≤50/class. Kernel: `smartflat_repro`.
Set env `BPI_SMOKE=1` for a fast plumbing check (tiny caps/shuffles).

In [ ]:
%load_ext autoreload
%autoreload 2
import os
os.environ.setdefault('NUMBA_THREADING_LAYER', 'workqueue')  # fork-safe rTWE under nbconvert
os.environ.setdefault('MPLBACKEND', 'agg')                   # headless figures
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display

from smartflat.utils.utils_io import get_data_root
from smartflat.utils.utils import upsample_sequence
from smartflat.features.symbolic_barycenter.generalization.bpi import (
    download_bpi, load_bpi, build_bpi_ground_cost, bpi_activity_mapping, DATASETS)
from smartflat.features.symbolic_barycenter.generalization.suite import run_generalization_suite
from smartflat.features.symbolic_barycenter.generalization.headroom import (
    headroom_table, restrict_to_shared_vocabulary, pooled_markov_surrogate)
from smartflat.features.symbolic_barycenter.registries import default_baseline_methods
from smartflat.features.symbolic_barycenter.visualization import plot_cohort_barycenters

pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)

NAME = 'bpi2012'
SMOKE = os.environ.get('BPI_SMOKE') == '1'
# UPSAMPLE ~ captures the length distribution (median 5, mean 11, p90 27); order-null uses ragged raw.
CAP_ORD, N_SH_MAIN, N_SH_DECOY, N_SH_CTRL = (40, 5, 5, 5) if SMOKE else (200, 100, 100, 50)
CAP_QUAL, UPSAMPLE = (8, 12) if SMOKE else (50, 32)
OUT = os.path.join(get_data_root(), 'outputs', 'symbolic_barycenter', 'generalization', NAME)
os.makedirs(OUT, exist_ok=True)
print('dataset:', NAME, '| SMOKE:', SMOKE, '| output dir:', OUT)

In [ ]:
# --- notebook-local helpers (presentation layer) ---
def length_match(Xr, lr, n_bins=12, rng=None):
    '''Subsample so both classes share the same trace-length distribution (per length bin).'''
    rng = rng if rng is not None else np.random.default_rng(0)
    lr = np.asarray(lr); lens = np.array([len(x) for x in Xr]); cls = sorted(set(lr))
    edges = np.unique(np.quantile(lens, np.linspace(0, 1, n_bins + 1)))
    binid = np.clip(np.digitize(lens, edges[1:-1]), 0, len(edges) - 2)
    keep = []
    for b in range(binid.max() + 1):
        idx = [np.where((binid == b) & (lr == c))[0] for c in cls]
        m = min(len(i) for i in idx)
        for i in idx:
            keep.extend(rng.choice(i, m, replace=False).tolist() if m else [])
    keep = sorted(keep)
    return [Xr[i] for i in keep], lr[keep]

def cap_per_class(X, y, cap, rng):
    '''Random subsample to at most `cap` sequences per class.'''
    y = np.asarray(y); keep = []
    for c in sorted(set(y)):
        idx = np.where(y == c)[0]
        keep.extend((rng.choice(idx, cap, replace=False) if len(idx) > cap else idx).tolist())
    keep = sorted(keep)
    return [X[i] for i in keep], y[keep]

def mean_transition(seqs, Gc):
    '''Per-class mean of per-sequence L1-normalized bigram-transition matrices.'''
    M = np.zeros((Gc, Gc))
    for s in seqs:
        s = np.asarray(s, int); T = np.zeros((Gc, Gc))
        for a, b in zip(s[:-1], s[1:]):
            T[a, b] += 1
        if T.sum() > 0:
            T /= T.sum()
        M += T
    return M / max(len(seqs), 1)

## 1. Load + config-vs-real-files sanity (vs published)

In [ ]:
download_bpi(NAME)                       # flag-guarded single-file fetch (3.3 MB)
meta, X, labels, G = load_bpi(NAME)
cfg = DATASETS[NAME]
lens = np.array([len(x) for x in X])
display(pd.DataFrame({
    'metric': ['N traces (kept)', '|V| = G (incl. padding 0)', 'V activities (guarded)',
               '#outcome classes', 'trace-length p10/50/90', 'published N', 'published #classes'],
    'value':  [len(X), G, G - 1, len(set(labels)),
               tuple(np.percentile(lens, [10, 50, 90]).round(1)),
               cfg['n_traces'], cfg['n_classes']],
}))
print('per-class outcome counts:', dict(Counter(labels)))
print('length mean/min/max:', round(lens.mean(), 1), int(lens.min()), int(lens.max()))
print('note: 23 distinct concept:name activities; 20 after COMPLETE-filter + removing the 3 '
      'outcome markers. Published "|V|=36" counts activity x lifecycle; N=13087 is the strong check '
      '(399 traces have no terminal outcome and are dropped -> 12688 kept).')

## 2. Co-occurrence ground cost

In [ ]:
D_G = build_bpi_ground_cost(NAME, X=X, kind='cooccurrence')
assert D_G.shape == (G, G), (D_G.shape, G)
assert np.allclose(D_G, D_G.T) and np.allclose(np.diag(D_G), 0.0)
print('D_G shape:', D_G.shape, '| symmetric + zero-diagonal OK')

## 3. Frequency-headroom GATE (full vocab + shared vocabulary)

`hist_auc` is the out-of-fold AUC of an L1-normalized **unigram histogram** — the frequency channel
the order-shuffle holds fixed. `band='saturated'` (≥0.95) means the null has **zero headroom**: order
cannot add anything a frequency classifier does not already have. This is the gate that stops us from
repeating Breakfast's 6.8 h vacuous run.

In [ ]:
scr_full = headroom_table(X, labels, G, D_G, name=NAME)
cols = ['comparison', 'n', 'n_class0', 'n_class1', 'hist_auc', 'w_hist_dist',
        'vocab_jaccard', 'n_shared', 'headroom_band']
display(scr_full[[c for c in cols if c in scr_full.columns]].round(3))

classes = sorted(set(labels))
pairs = [(classes[i], classes[j]) for i in range(len(classes)) for j in range(i + 1, len(classes))]
rows = []
for pair in pairs:
    Xr, lr, Gc, Dc, info = restrict_to_shared_vocabulary(X, labels, pair, D_G=D_G)
    s = headroom_table(Xr, lr, Gc, Dc, name=f'{pair[0]}_vs_{pair[1]}_shared').iloc[0]
    rl = np.array([len(x) for x in Xr])
    rows.append(dict(pair=f'{pair[0]}_vs_{pair[1]}', G_c=Gc, n=len(Xr),
                     len_p50=float(np.percentile(rl, 50)), hist_auc=float(s['hist_auc']),
                     band=s['headroom_band']))
print('\nshared-vocabulary headroom (class-exclusive markers dropped):')
display(pd.DataFrame(rows).round(3))
print('=> full vocab AND shared vocab are SATURATED: frequency dominates. '
      'BPI-2012 joins Breakfast, now cross-domain and at scale.')

## 3b. Depth — does controlling trace LENGTH open any order headroom?

Declined applications are structurally short (median ≈ 3 events), so **trace length** (≈ the total
activity count) trivially separates them. We subsample each shared-vocabulary pair to a common
length distribution and re-screen. A pair that drops out of `saturated` into `sweet`
(0.55 < `hist_auc` < 0.95) is a genuine **frequency-AND-length-controlled** order-test candidate.

In [ ]:
lm_rows, sweet_pairs, lm_hist = [], [], {}
for pair in pairs:
    Xr, lr, Gc, Dc, info = restrict_to_shared_vocabulary(X, labels, pair, D_G=D_G)
    Xm, lm = length_match(Xr, lr, rng=np.random.default_rng(0))
    s = headroom_table(Xm, lm, Gc, Dc, name=f'{pair[0]}_vs_{pair[1]}_lenmatched').iloc[0]
    rl = np.array([len(x) for x in Xm])
    band = s['headroom_band']
    lm_hist[pair] = float(s['hist_auc'])          # frequency baseline the order feature must beat
    lm_rows.append(dict(pair=f'{pair[0]}_vs_{pair[1]}', n=len(Xm), Gc=Gc,
                        len_p50=float(np.percentile(rl, 50)), hist_auc=float(s['hist_auc']),
                        w_hist_dist=float(s['w_hist_dist']), band=band))
    if band == 'sweet':
        sweet_pairs.append(pair)
display(pd.DataFrame(lm_rows).round(3))
print('sweet (length-controlled headroom) pairs -> order-null runs on these:', sweet_pairs)

## 4. Order-null on the length-controlled sweet pairs + triple gate

For each sweet pair (shared vocabulary, length-matched, ≤`CAP_ORD`/class):
- **main** — `run_generalization_suite` order-null over the 4 feature×shuffle combos.
- **decoy** — `pooled_markov_surrogate` (per-sequence multiset preserved, order resampled from a
  *class-pooled* Markov model): a well-calibrated harness must return `order_helps=False` (type-I).
- **control** — permuted labels: `auc_intact` must collapse to ≈0.5.
- **mechanism** — top class-differentiating bigram transitions (interpretable only if order helps).

**Critical guard against the anti-conservative shuffle-null.** `order_helps == (ci_low > 0)` only
says the *transition classifier* loses AUC when shuffled — it does **not** say order beats *frequency*.
So the verdict `ORDER_BEYOND_FREQ` additionally requires `beats_freq` = the order feature's
`auc_intact` exceeds the length-matched unigram-histogram baseline `hist_auc_freq`. Order is claimed to
help beyond frequency **only** if `beats_freq` **and** main fires **and** decoy stays null **and**
control ≈ 0.5.

In [ ]:
name2id = bpi_activity_mapping(NAME)
id2name = {v: k for k, v in name2id.items()}

def order_table(Xo, yo, Go, Do, tag, n_sh):
    return run_generalization_suite(Xo, yo, Go, Do, name=tag,
                                    run_quality=False, n_shuffles=n_sh)['order']

order_rows, gate_rows, mech = [], [], {}
for pair in sweet_pairs:
    tag = f'{pair[0]}_vs_{pair[1]}'
    Xr, lr, Gc, Dc, info = restrict_to_shared_vocabulary(X, labels, pair, D_G=D_G)
    Xm, lm = length_match(Xr, lr, rng=np.random.default_rng(0))
    Xc, lc = cap_per_class(Xm, lm, CAP_ORD, np.random.default_rng(1))

    main = order_table(Xc, lc, Gc, Dc, tag, N_SH_MAIN).assign(pair=tag, kind='main')
    decoy = order_table(pooled_markov_surrogate(Xc, Gc, random_state=0), lc, Gc, Dc,
                        tag, N_SH_DECOY).assign(pair=tag, kind='decoy')
    ctrl = order_table(Xc, np.random.default_rng(7).permutation(lc), Gc, Dc,
                       tag, N_SH_CTRL).assign(pair=tag, kind='control')
    order_rows += [main, decoy, ctrl]
    gate_rows.append(dict(pair=tag, n=len(Xc),
                          hist_auc_freq=lm_hist[pair],
                          order_auc_intact=float(main['auc_intact'].max()),
                          beats_freq=bool(main['auc_intact'].max() > lm_hist[pair]),
                          main_helps=int(main['order_helps'].sum()),
                          main_max_delta=float(main['delta_auc'].max()),
                          main_min_p=float(main['p_perm'].min()),
                          decoy_helps=int(decoy['order_helps'].sum()),
                          ctrl_auc_intact=float(ctrl['auc_intact'].mean())))
    # mechanism: top class-differentiating transitions on the length-matched compact alphabet
    comp2name = {c: id2name.get(o, f'id{o}') for o, c in info['remap'].items()}
    c0, c1 = pair
    D = mean_transition([Xc[i] for i in np.where(lc == c0)[0]], Gc) - \
        mean_transition([Xc[i] for i in np.where(lc == c1)[0]], Gc)
    flat = sorted(((abs(D[a, b]), a, b, D[a, b]) for a in range(Gc) for b in range(Gc)),
                  reverse=True)[:8]
    mech[tag] = pd.DataFrame([dict(transition=f'{comp2name.get(a,a)} -> {comp2name.get(b,b)}',
                                   delta_freq=round(d, 4), favors=(c0 if d > 0 else c1))
                              for _, a, b, d in flat])

order_df = pd.concat(order_rows, ignore_index=True)
order_df.to_csv(os.path.join(OUT, 'order_null_lenmatched.csv'), index=False)
gate_df = pd.DataFrame(gate_rows)
# order beats frequency ONLY if the order feature exceeds the frequency baseline AND the shuffle
# null fires AND the calibration decoy stays null AND the negative control collapses to ~0.5.
gate_df['ORDER_BEYOND_FREQ'] = (gate_df['beats_freq'] & (gate_df['main_helps'] > 0)
                                & (gate_df['decoy_helps'] == 0)
                                & gate_df['ctrl_auc_intact'].between(0.4, 0.6))
gate_df.to_csv(os.path.join(OUT, 'order_gate_summary.csv'), index=False)
print('GATE SUMMARY — order beats frequency only if beats_freq & main_helps & !decoy & ctrl~0.5:')
display(gate_df.round(3))
print('\nmain-null detail (per sweet pair x 4 combos):')
display(order_df[order_df['kind'] == 'main'][
    ['pair', 'feature', 'shuffle', 'auc_intact', 'auc_null_mean', 'delta_auc',
     'ci_low', 'p_perm', 'order_helps']].round(3))
for tag, m in mech.items():
    print(f'\ntop class-differentiating transitions — {tag}:')
    display(m)

## 5. Representation quality (all 3 outcomes, ≤`CAP_QUAL`/class subsample)

In [ ]:
rng = np.random.default_rng(42)
Xq, labelsq = cap_per_class(X, labels, CAP_QUAL, rng)
res_q = run_generalization_suite(Xq, np.asarray(labelsq, dtype=object), G, D_G,
                                 name=NAME, run_order=False, upsample_to=UPSAMPLE)
res_q['quality'].reset_index().to_csv(os.path.join(OUT, 'quality.csv'), index=False)
print(f'quality on {len(Xq)} sequences (<= {CAP_QUAL}/class), upsampled to {UPSAMPLE}:')
display(res_q['quality'].round(3))

## 6. Prototypical execution per outcome (chronograms)

In [ ]:
methods = default_baseline_methods(D_G)
code_to_label = {i: n for n, i in name2id.items()}   # 0 (padding) intentionally absent -> masked
CAP_CHRONO, rng = (8 if SMOKE else 60), np.random.default_rng(0)
proto = {}
for a in sorted(set(labels)):
    ia = np.where(labels == a)[0]
    if len(ia) > CAP_CHRONO:
        ia = rng.choice(ia, CAP_CHRONO, replace=False)
    Xa = np.vstack([upsample_sequence(X[i], UPSAMPLE) for i in ia]).astype(int)
    proto[a] = np.asarray(methods['edit_median']['build'](Xa, 0)).astype(int)
plot_cohort_barycenters(proto, groups=sorted(proto), code_to_label=code_to_label,
                        mask_background=True,
                        title=f'{NAME}: prototypical execution per outcome (edit-median barycenter)',
                        savepath=os.path.join(OUT, 'chronograms.png'))
print('saved', os.path.join(OUT, 'chronograms.png'))

## 7. Result

**Frequency dominates BPI-2012 at face value.** The unigram histogram already classifies the loan
outcome near-perfectly on the full vocabulary (`hist_auc` = 1.00 / 1.00 / 0.99) and on the shared
vocabulary (≥ 0.97). The hoped-for "one vocabulary, differs mainly in order" premise is **empirically
false**: outcomes differ in activity *frequency* and *trace length* (declined applications are short,
median ≈ 3 events). BPI-2012 therefore **replicates** the SDS2/Breakfast frequency-dominance finding in
a new domain (process mining) and at scale (12,688 traces) — evidence for §7.2, and it validates
computational feasibility (§2.2; all six baselines run, see §5).

**Length-controlled order test — order does NOT beat frequency (`ORDER_BEYOND_FREQ` = False, both
pairs).** Matching trace length opens headroom on the two "declined" contrasts, a genuine
frequency-AND-length-controlled order test. The bounded order-null (≤200/class, `n_shuffles`=100) is
negative for both, for two *different* reasons that vindicate the gates:
- **A_APPROVED_vs_A_DECLINED:** the shuffle-null fires (`order_helps` 4/4) with a clean decoy and
  control (≈0.51) — yet the order feature's AUC (0.71) is **below** the frequency baseline (0.74). The
  §15 shuffle-null is anti-conservative; the frequency-baseline guard correctly rejects the "positive".
- **A_CANCELLED_vs_A_DECLINED:** the order feature only ties frequency (0.902 vs 0.901) **and** the
  calibration decoy *also* fires (`decoy_helps` 4/4) — the null is type-I-inflated here, so `order_helps`
  is a harness artifact, not order information.

**Takeaway.** Even a constructed, frequency-*and*-length-controlled task from a real process log carries
no order signal beyond frequency — the strongest cross-domain confirmation yet of the paper's thesis,
and a clean empirical demonstration that the frequency-baseline + calibration-decoy gates are necessary
(the raw shuffle-null would have reported two spurious positives). Scope-limited constructed contrast;
does not overturn the SDS2 negative.

*Not yet folded into `RESULTS_HANDOFF_barycenters.md` (RH ends at §26). Artifacts (git-ignored `OUT/`):
`order_gate_summary.csv`, `order_null_lenmatched.csv`, `quality.csv`, `chronograms.png`.*